Implement HITL for a food ordering application.

HITL Points

-   Confirm restaurant for the item(s), user is asking for.
-   Confirm the order details like restaurant vs item, quantity of each item, user address and total price of items to be ordered.
-   Cancel order

Specifications that needs to be taken care of whille implementing HITL.

- We have list of restaurant where each record represents a retraunt. Our job is when the user asks for the respective food item(s) then we need suggest the available restaurant for the items asked "OR", if the user asked for the restaurant, then we need to show the list for avaliable food items at the restaurant.

- While creating the order user should be able to select the restaurant for the asked item(s).

- The workflow should explicitely ask the user for confirmation before placing the order.

- The workflod should explicitely ask the user for confirmation before cancelling the order.

- Before placing the order at any point of time if the user wants to go back to any of the specified step to make changes then he should be able to do so, but after placing the order user should not be allowed to make modifications.

In [3]:
from dotenv import load_dotenv
from pathlib import Path
import os

env_path = Path.cwd().parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")

In [4]:
load_dotenv(env_path)
api_key = os.getenv("OPENAI_API_KEY")

In [6]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command
from langchain_core.tools import tool
from rich import print
import uuid

In [7]:
from langchain.agents.structured_output import ProviderStrategy

In [8]:
# Dummy hardcoaded data for the list of restaurant.

restaurant_list = [
    {
        "name": "Royal Rajasthan",
        "cuisine": "Rajasthani",
        "location": "Downtown",
        "food_items": [
            {
                "name": "Dal Baati Churma",
                "category": "Main Course",
                "price": 320
            },
            {
                "name": "Gatte Ki Sabzi",
                "category": "Main Course",
                "price": 240
            },
            {
                "name": "Ker Sangri",
                "category": "Vegetarian",
                "price": 220
            },
            {
                "name": "Pyaaz Kachori",
                "category": "Starter",
                "price": 100
            },
            {
                "name": "Ghevar",
                "category": "Dessert",
                "price": 150
            }
        ]
    },

    {
        "name": "Coastal Spice",
        "cuisine": "South Indian",
        "location": "Green Park",
        "food_items": [
            {
                "name": "Masala Dosa",
                "category": "Main Course",
                "price": 180
            },
            {
                "name": "Idli Sambar",
                "category": "Breakfast",
                "price": 120
            },
            {
                "name": "Medu Vada",
                "category": "Starter",
                "price": 110
            },
            {
                "name": "Uttapam",
                "category": "Main Course",
                "price": 160
            },
            {
                "name": "Payasam",
                "category": "Dessert",
                "price": 130
            }
        ]
    },

    {
        "name": "Punjab House",
        "cuisine": "Punjabi",
        "location": "Connaught Place",
        "food_items": [
            {
                "name": "Amritsari Kulcha",
                "category": "Main Course",
                "price": 220
            },
            {
                "name": "Chole Bhature",
                "category": "Main Course",
                "price": 200
            },
            {
                "name": "Sarson Da Saag",
                "category": "Vegetarian",
                "price": 260
            },
            {
                "name": "Tandoori Chicken",
                "category": "Starter",
                "price": 350
            },
            {
                "name": "Phirni",
                "category": "Dessert",
                "price": 140
            }
        ]
    },
    {
    "name": "Curry Corner",
    "cuisine": "Indian",
    "location": "Midtown",
    "food_items": [
        {
            "name": "Butter Chicken",
            "category": "Main Course",
            "price": 350,
            "available": True
        },
        {
            "name": "Paneer Tikka",
            "category": "Starter",
            "price": 280,
            "available": True
        },
        {
            "name": "Biryani",
            "category": "Main Course",
            "price": 300,
            "available": True
        },
        {
            "name": "Dal Makhani",
            "category": "Main Course",
            "price": 220,
            "available": True
        },
        {
            "name": "Naan",
            "category": "Bread",
            "price": 60,
            "available": True
        }
    ]
}
]

In [9]:
# Configuring the chatmodel, which is suppoed to be used during the workflow.

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key,
    temperature=0
)

In [10]:
# class representing the asked item(s) asked by user.

class RequestedItem(BaseModel):
    food_item: str | None = Field(
        default=None,
        description="Contains the name of the food item(s) the user is asking for."
    )

In [11]:
# class representing the asked item(s) asked by user.

class Requestedrestaurant(BaseModel):
    restaurant: str | None = Field(
        default = None,
         description="Contains the name of the restaurant(s) the user is asking for."
    )

In [12]:
class RequestedEntry(BaseModel):

    food_item: str | None = Field(
        default=None,
        description="Requested food item."
    )

    restaurant: str | None = Field(
        default=None,
        description="Requested restaurant."
    )

In [13]:
# Class representing the complete list of the requested item and requested restaurant.

class RequestedFoodAndRestaurantCatelog(BaseModel):
    requested_catelog_item: list[RequestedEntry] = Field(
        ...,
        description="List containing the requested food items and restaurants."
    )

In [14]:
# class representing item to be ordered 
class ItemToBeOrdered(BaseModel):
    item_name: str = Field(..., description="Name of the food item")
    quantity: int = Field(..., description="Quantity of this item")
    unit_price: float = Field(..., description="price of the item")

# class representing the restaurant from which the requested item is supposed to be order
class restaurantToBeOrderedFrom(BaseModel):
    restaurant_name: str = Field(..., description="Name of the restaurant")
    restaurant_localtion: str = Field(..., description="Location of the restaurant")
    items: list [ItemToBeOrdered] = Field(..., description="list of items to be ordered from an individual restaurant")

# class representing the complete details of the requested order.
class OrderDetails(BaseModel):
    requested_order_catelog : list[restaurantToBeOrderedFrom] = Field(..., description="contains the order catelog of the requested items")
    total_price: float = Field(..., description="Total order price")

In [15]:
from typing import Annotated
from operator import add

class State(BaseModel):
    request_catalog: Annotated[
        list[RequestedFoodAndRestaurantCatelog],
        add
    ]

    confirm_order_details: Annotated[
        list[OrderDetails],
        add
    ]
    

In [21]:
# function to check whether requested item is available in the restaurant.
def map_requested_item_vs_restaurant(requested_item: str, restaurant: dict) -> bool:
    for food_item in restaurant.get("food_items", []):
        if food_item.get("name", "").strip().lower() == requested_item.strip().lower():
            return True
    return False


# Tool for suggesting the restaurant(s) for the requested item
@tool
def suggest_restaurants_for_the_requested_item(requested_item: str) -> list:
    """
        This tool is used to suggest the name of restaurants for the requested food item by user
    """
    restaurant_list = globals().get("restaurant_list", [])

    suggested_restaurants = []
    
    for restaurant in restaurant_list:
        if map_requested_item_vs_restaurant(requested_item, restaurant):
            suggested_restaurants.append(restaurant)

    return suggested_restaurants


In [22]:
# Tool for suggesting the food item(s) for the requested restaurant
@tool
def suggest_food_item_for_the_requested_restaurant(requested_restaurant: str) -> list:
    """
        This tool is used to suggest the food items that a restaurant is offering.
    """
    restaurant_list = globals().get("restaurant_list", [])

    suggested_list_of_food_items = []
    for restaurant in restaurant_list:
        if restaurant.get("name", "").strip().lower() == requested_restaurant.strip().lower():
            suggested_list_of_food_items.extend(restaurant.get("food_items", []))
            break

    return suggested_list_of_food_items


In [23]:
# Tool to check whether the explicitly requested item belongs to an explicitly requested restaurant

@tool
def cross_check_the_restaurant_with_the_requested_food_item(requested_item: str, requested_restaurant: str) -> list:
    """
        If the user is explicitly requesting an item from a specific restaurant,
        then this tool searches the restaurant list and find if the requested item
        belongs to the requested restaurant or not, if it belongs then simply it
        returns the details of the restaurant
    """
    
    restaurant_list = globals().get("restaurant_list", [])

    suggested_list = []

    for restaurant in restaurant_list:
        if restaurant.get("name", "").strip().lower() != requested_restaurant.strip().lower():
            continue
        else:
            for food_item in restaurant.get("food_items", []):
                if food_item.get("name", "").strip().lower() == requested_item.strip().lower():
                    suggested_list.append(restaurant)

    return suggested_list


In [ ]:
# Agent

In [24]:
# Agent for identification of the food item and restaurant.
from langchain.agents.structured_output import ToolStrategy


identify_food_and_resturant_agent = create_agent(
    model =llm,
    response_format=ProviderStrategy(RequestedFoodAndRestaurantCatelog),
    tools =[
        suggest_restaurants_for_the_requested_item,
        suggest_food_item_for_the_requested_restaurant,
        cross_check_the_restaurant_with_the_requested_food_item
    ],
    system_prompt="You are a food and restaurant identification agent. Your task is to identify the food items and restaurants from the user's query. You have some tools if the user asks to suggest some food items or resturant then use the appropriate tool for it."
)

result = identify_food_and_resturant_agent.invoke({
    "messages": [("user", "")]
})
print(result["messages"][-1].content)

In [25]:
result = identify_food_and_resturant_agent.invoke({
    'messages':[
        {
            "role": "user",
            "content": "Suggest me some good restaurants for Dal Makhani."
        }
    ]
})

In [27]:
print(result)
print(result["messages"][-1].content)

{
    'messages': [
        HumanMessage(
            content='Suggest me some good restaurants for Dal Makhani.',
            additional_kwargs={},
            response_metadata={},
            id='0d869509-3347-4dc0-91df-83bc2fa4c2fb'
        ),
        AIMessage(
            content='',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 24,
                    'prompt_tokens': 370,
                    'total_tokens': 394,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
                },
                'model_provider': 'openai',
                'model_name': 'gpt-4.1-mini-2025-04-14',
                'system_fingerprint': 'fp_4ffa17cc03',
                'id': 'chatcmpl-EeqIyrfa7fTSOnzQBKBO0xULA6eMw',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a18f24-591c-7203-b6ff-8073c7ae030b-0',
            tool_calls=[
                {
                    'name': 'suggest_restaurants_for_the_requested_item',
                    'args': {'requested_item': 'Dal Makhani'},
                    'id': 'call_w2DeOWaDERRk1eLv5FZCT32D',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 370,
                'output_tokens': 24,
                'total_tokens': 394,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='[{"name": "Curry Corner", "cuisine": "Indian", "location": "Midtown", "food_items": [{"name": 
"Butter Chicken", "category": "Main Course", "price": 350, "available": true}, {"name": "Paneer Tikka", "category":
"Starter", "price": 280, "available": true}, {"name": "Biryani", "category": "Main Course", "price": 300, 
"available": true}, {"name": "Dal Makhani", "category": "Main Course", "price": 220, "available": true}, {"name": 
"Naan", "category": "Bread", "price": 60, "available": true}]}]',
            name='suggest_restaurants_for_the_requested_item',
            id='ecd7fd4b-3149-4011-8273-d5f2c94a7120',
            tool_call_id='call_w2DeOWaDERRk1eLv5FZCT32D'
        ),
        AIMessage(
            content='{"requested_catelog_item":[{"food_item":"Dal Makhani","restaurant":"Curry Corner"}]}',
            additional_kwargs={'parsed': None, 'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 32,
                    'prompt_tokens': 564,
                    'total_tokens': 596,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0,
                        'text_tokens': None
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cache_write_tokens': None,
                        'cached_tokens': 0,
                        'image_tokens': None,
                        'text_tokens': None
                    }
      

{"requested_catelog_item":[{"food_item":"Dal Makhani","restaurant":"Curry Corner"}]}